##Silver Transformation of the results bronze table
### 1. Read the table from bronze schema

### Getting the batch id as input parameter

In [0]:
dbutils.widgets.text("p_batch_id","")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00.Common/01.Environment-config

In [0]:
%run "../00.Common/03.Helper_Notebook_Silver"

In [0]:
source_name = f"{catalog_name}.{bronze_schema}.results"
target_name = f"{catalog_name}.{silver_schema}.results"

In [0]:
#import the sql function and filter the df with the batch id
from pyspark.sql import functions as F
results_df = (spark.table(source_name).filter(F.col("batch_id")==v_batch_id))
display(results_df)

### 2. Select the required columns for the analytics

In [0]:
results_clean_df = (results_df.select(
    F.col("date"),
    F.col("raceName"),
    F.col("round"),
    F.col("season"),
    F.col("constructorId"),
    F.col("driverId"),
    F.col("grid"),
    F.col("laps"),
    F.col("number"),
    F.col("points"),
    F.col("position"),
    F.col("positionText"),
    F.col("status"),
    F.col("ingestion_timestamp"),
    F.col("SourceFile"),
    F.col("batch_id")
    )
    .withColumnsRenamed({"raceName":"race_name","constructorId":"constructor_id","driverId":"driver_id","date":"race_date","grid":"grid_position","laps":"completed_laps","number":"car_number","position":"final_position","positionText":"final_position_text"})
                    
                       )
display(results_clean_df)                       

### 4.Remove the null values on the primary keys and dropping duplicates

In [0]:
results_valid_df = (results_clean_df
                    .filter(F.col("season").isNotNull()&F.col("round").isNotNull()&F.col("constructor_id").isNotNull()&F.col("driver_id").isNotNull())
                    .dropDuplicates(["season","round","constructor_id","driver_id"]))


In [0]:
#Checking the number of duplicates removed
display(results_clean_df.count()-results_valid_df.count())

### 5. Transforming the column values 

In [0]:
# Converting the values to initcap format in locality and circuit name columns
results_final_df = (results_valid_df
                     .withColumn("race_name",F.initcap(F.col("race_name")))
)
display(results_final_df)

In [0]:
results_final_df.columns

### 6. Writing the final dataframe as table into the silver schema

In [0]:
write_to_silver(
    input_df = results_final_df,
    target_table = target_name,
    merge_condition = "t.season=s.season AND t.round=s.round AND t.constructor_id=s.constructor_id AND t.driver_id=s.driver_id",
    columns_to_update = ['race_date',
 'race_name',
 'round',
 'season',
 'constructor_id',
 'driver_id',
 'grid_position',
 'completed_laps',
 'car_number',
 'points',
 'final_position',
 'final_position_text',
 'status',
 'ingestion_timestamp',
 'SourceFile',
 'batch_id']
)

In [0]:
%sql
select * from formula1_incr.silver.results;